# Deep research (notebook MVP)

### Imports and environment


In [ ]:
import asyncio
import os
from datetime import datetime

import httpx
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from agents import Agent, Runner, function_tool, trace

load_dotenv(find_dotenv())


### Constants


In [ ]:
MODEL_DEFAULT = "gpt-4o-mini"
MODEL_PLANNER = "gpt-5-nano"

MAX_RESEARCH_TURNS = 20
DEFAULT_PHRASES_COUNT = 5
MAX_TRIAGE_ROUNDS = 3
MAX_EVALUATOR_RETRIES = 2

PUSHOVER_MESSAGES_URL = "https://api.pushover.net/1/messages.json"
PUSHOVER_NOTIFY_MESSAGE = "Research job finished"



### Pydantic models


In [ ]:
class TriageResults(BaseModel):
    is_ambiguous: bool = Field(description="Whether the query is ambiguous")
    what_is_ambiguous: list[str] = Field(
        description="List of things that are ambiguous in the query",
    )


class ClarificationQuestion(BaseModel):
    question: str = Field(description="One focused follow-up question to resolve ambiguity")


class SearchItem(BaseModel):
    query: str = Field(description="The search term to use")
    reason: str = Field(description="Why this search is important")


class SearchPlan(BaseModel):
    searches: list[SearchItem] = Field(description="List of web searches to perform")


class ResearchItem(BaseModel):
    query: str = Field(description="The search term to use")
    reason: str = Field(description="Why this search is important")
    result: str = Field(description="The result of the search")


class ResearchData(BaseModel):
    results: list[ResearchItem] = Field(description="Results of the web searches")


class Report(BaseModel):
    markdown_report: str = Field(description="Markdown formatted body of the report.")
    summary: str = Field(description="Summary addressing original question.")


class EvaluationResult(BaseModel):
    passes: bool = Field(description="Whether the report is acceptable.")
    gaps: list[str] = Field(
        default_factory=list,
        description="Concrete shortcomings if passes is false.",
    )
    suggested_searches: list[str] = Field(
        default_factory=list,
        description="Web search queries to improve the report if passes is false.",
    )


### Serper client


In [ ]:
async def serper_search(query: str) -> str:
    async with httpx.AsyncClient(timeout=30.0) as client:
        r = await client.post(
            "https://google.serper.dev/search",
            json={"q": query},
            headers={"X-API-KEY": os.environ["SERPER_API_KEY"]},
        )
        r.raise_for_status()
        data = r.json()
    organic = data.get("organic") or []
    parts: list[str] = []
    for i, item in enumerate(organic[:10], 1):
        title = item.get("title") or ""
        snippet = item.get("snippet") or ""
        link = item.get("link") or ""
        parts.append(f"{i}. {title}\n{snippet}\n{link}")
    return "\n\n".join(parts) if parts else str(data)[:8000]



### Triage


In [ ]:
def _triage_instruction() -> str:
    return """You are query Triager tasked with inspecting the query for ambiguity.

If the query is clear and unambiguous, return TriageResults with is_ambiguous set to False.
If the query is ambiguous, return TriageResults with is_ambiguous set to True and a list of things that are ambiguous in the query.
"""


def _triage_agent() -> Agent:
    return Agent(
        name="Triage Agent",
        instructions=_triage_instruction(),
        output_type=TriageResults,
        model=MODEL_DEFAULT,
    )


async def run_triage(canonical_question: str) -> TriageResults:
    print("[Triage] checking query...")
    result = await Runner.run(_triage_agent(), f"# Query\n{canonical_question}")
    out = result.final_output
    if out.is_ambiguous:
        print("[Triage] ambiguous - needs clarification")
    else:
        print("[Triage] clear - proceeding")
    return out



### Clarifier


In [ ]:
def _clarifier_instruction() -> str:
    return """You are a Clarification Agent. Given an ambiguous query and a list of ambiguous aspects,
output exactly ONE focused follow-up question the user should answer. Be concise.
"""


def _clarifier_agent() -> Agent:
    return Agent(
        name="Clarifier Agent",
        instructions=_clarifier_instruction(),
        output_type=ClarificationQuestion,
        model=MODEL_DEFAULT,
    )


async def run_clarifier(canonical_question: str, aspects: list[str]) -> str:
    print("[Clarifier] drafting one follow-up question...")
    payload = (
        f"# Original Query\n{canonical_question}\n\n"
        f"# Ambiguous Aspects Found\n{aspects}\n\n"
        "Respond with exactly one follow-up question."
    )
    result = await Runner.run(_clarifier_agent(), payload)
    print("[Clarifier] done")
    return result.final_output.question.strip()



### Planner


In [ ]:
def _planner_instruction() -> str:
    return """You are a research planning agent. Your task is to prepare a number of web search phases helpful to best answer the question.
Make sure to cover all the aspects of the question. Provide reason for each search phrase."""


def _planner_agent() -> Agent:
    return Agent(
        name="PlannerAgent",
        instructions=_planner_instruction(),
        output_type=SearchPlan,
        model=MODEL_PLANNER,
    )


async def run_planner(canonical_question: str, phrases_count: int) -> SearchPlan:
    print("[Planner] building search plan...")
    inp = (
        f"# Question\n{canonical_question}\n\n"
        f"Provide {phrases_count} web search phrases to best answer the question."
    )
    result = await Runner.run(_planner_agent(), inp)
    plan = result.final_output
    print(f"[Planner] done ({len(plan.searches)} searches)")
    return plan


### Research


In [ ]:
@function_tool
async def web_search(query: str) -> str:
    """Search the web and return a plain-text block of results."""
    q = query.strip()
    preview = q if len(q) <= 70 else q[:67] + "..."
    print(f"[Serper] {preview}")
    return await serper_search(query)


def _research_instruction() -> str:
    return f"""You are a research agent. Your task is to perform a number of web searches from the list of provided search topics.
You are given a curated list of search topics. For each item in the list, perform a web search for the query and produce a concise summary.

Keep summaries to 3-4 paragraphs and under 500 words. Use guidance from the reason clause for the search to produce succinct and to-the-point summary.
Copy the reason verbatim from input to the output.

# Date
{datetime.now().strftime('%Y-%m-%d')}
"""


def _research_agent() -> Agent:
    return Agent(
        name="ResearchAgent",
        instructions=_research_instruction(),
        output_type=ResearchData,
        tools=[web_search],
        model=MODEL_DEFAULT,
    )


async def run_research(plan: SearchPlan) -> ResearchData:
    print(f"[Research] agent run ({len(plan.searches)} planned searches)...")
    inp = f"# Search Topics\n{plan.model_dump_json()}"
    result = await Runner.run(
        _research_agent(),
        inp,
        max_turns=MAX_RESEARCH_TURNS,
    )
    print("[Research] done")
    return result.final_output



### Report


In [ ]:
def _report_instruction() -> str:
    return """You are Senior Researcher writing a comprehensive report. Your task is to generate markdown-formatted report based on the research materials.
You will be given:
 * Original question
 * Research materials containing results of prior searches

Your task is to synthesize the research materials into a comprehensive report that addresses the original question. Aim for 1000+ words long report.
Make sure that the report is:
 * Comprehensive and well-structured
 * Clearly written and easy to follow
 * If it references any measurable values, keep the units the same (convert if needed)
"""


def _report_agent() -> Agent:
    return Agent(
        name="Report Agent",
        instructions=_report_instruction(),
        output_type=Report,
        model=MODEL_DEFAULT,
    )


async def run_report(canonical_question: str, research: ResearchData) -> Report:
    print("[Report] writing draft...")
    inp = (
        f"# Original Question\n{canonical_question}\n\n"
        f"# Research Materials\n{research.model_dump_json()}"
    )
    result = await Runner.run(_report_agent(), inp)
    print("[Report] done")
    return result.final_output



### Evaluator


In [ ]:
def _evaluator_instruction() -> str:
    return """You are a strict research quality evaluator.

Given the user's original question, the research material gathered, and the draft markdown report:
- Decide whether the report **fully and accurately** answers the question using the evidence available.
- If it passes, set passes=true and use empty lists for gaps and suggested_searches.
- If it fails, set passes=false, list concrete **gaps** (missing coverage, weak sourcing, factual issues), and suggest **short web search queries** (suggested_searches) that would help fix the report.

Be concise: gaps are short bullet-style strings; suggested_searches are query strings only."""


def _evaluator_agent() -> Agent:
    return Agent(
        name="Evaluator Agent",
        instructions=_evaluator_instruction(),
        output_type=EvaluationResult,
        model=MODEL_DEFAULT,
    )


async def run_evaluator(
    canonical_question: str,
    report: Report,
    research: ResearchData,
) -> EvaluationResult:
    print("[Evaluator] reviewing draft...")
    inp = (
        f"# Original Question\n{canonical_question}\n\n"
        f"# Research Materials\n{research.model_dump_json()}\n\n"
        f"# Draft Report (markdown)\n{report.markdown_report}\n\n"
        f"# Summary\n{report.summary}\n"
    )
    result = await Runner.run(_evaluator_agent(), inp)
    ev = result.final_output
    if ev.passes:
        print("[Evaluator] pass")
    else:
        print("[Evaluator] fail - draft not acceptable (retry if rounds remain)")
    return ev



### Retry plan


In [ ]:
async def build_retry_search_plan(
    canonical_question: str,
    ev: EvaluationResult,
) -> SearchPlan:
    print("[Retry plan] building supplemental searches...")
    trimmed = [s.strip() for s in ev.suggested_searches if s.strip()]
    if trimmed:
        items = [
            SearchItem(
                query=q[:500],
                reason="Evaluator-suggested search to address report gaps.",
            )
            for q in trimmed[:DEFAULT_PHRASES_COUNT]
        ]
        print(f"[Retry plan] using {len(items)} evaluator-suggested queries")
        return SearchPlan(searches=items)
    gap_block = (
        "\n\n## Gaps the previous draft must address\n"
        + "\n".join(f"- {g}" for g in ev.gaps)
        if ev.gaps
        else ""
    )
    print("[Retry plan] replanning from evaluator gaps")
    return await run_planner(
        canonical_question + gap_block,
        DEFAULT_PHRASES_COUNT,
    )



### Pushover


In [ ]:
def send_pushover_completion_notice() -> None:
    print("[Pushover] sending notification...")
    requests.post(
        PUSHOVER_MESSAGES_URL,
        data={
            "token": os.environ["PUSHOVER_TOKEN"],
            "user": os.environ["PUSHOVER_USER"],
            "message": PUSHOVER_NOTIFY_MESSAGE,
        },
        timeout=30.0,
    ).raise_for_status()
    print("[Pushover] sent")



### Merge research


In [ ]:
def merge_research(a: ResearchData, b: ResearchData) -> ResearchData:
    merged = ResearchData(results=list(a.results) + list(b.results))
    print(f"[Merge] {len(a.results)} + {len(b.results)} -> {len(merged.results)} results")
    return merged


### Orchestration


In [ ]:
async def deep_research(query: str, user_clarifications: str = "") -> None:
    with trace("Deep Research"):
        print("[Pipeline] start")
        q = query.strip()
        extra = user_clarifications.strip()
        if extra:
            canonical = q + "\n\n### User clarifications\n" + extra
        else:
            canonical = q

        t = await run_triage(canonical)
        if t.is_ambiguous:
            n_clar = extra.count("### Clarification")
            if n_clar >= MAX_TRIAGE_ROUNDS - 1:
                print(
                    "[Triage] abort - still ambiguous after max triage rounds; "
                    "not starting planner/research."
                )
                return
            pending_q = await run_clarifier(canonical, t.what_is_ambiguous)
            print(f"""--- Clarification question ---
{pending_q}

Append to the second argument a block like:

### Clarification
**Q:** {pending_q}
**A:** <your answer>

Then rerun with the same query and the longer user_clarifications string.
[Pipeline] paused for clarification""")
            return

        print("[Pipeline] entering research / report / evaluate loop")
        total_rounds = MAX_EVALUATOR_RETRIES + 1
        research_accum: ResearchData | None = None
        last_eval: EvaluationResult | None = None

        for round_idx in range(total_rounds):
            print(f"[Pipeline] round {round_idx + 1}/{total_rounds}")
            if round_idx == 0:
                plan = await run_planner(canonical, DEFAULT_PHRASES_COUNT)
            else:
                assert last_eval is not None
                plan = await build_retry_search_plan(canonical, last_eval)

            delta = await run_research(plan)
            research_accum = delta if research_accum is None else merge_research(research_accum, delta)

            report = await run_report(canonical, research_accum)
            ev = await run_evaluator(canonical, report, research_accum)

            if ev.passes:
                print("[Pipeline] evaluator approved - showing report")
                await asyncio.to_thread(send_pushover_completion_notice)
                display(Markdown(report.markdown_report))
                print("[Pipeline] done")
                return

            if round_idx == total_rounds - 1:
                print("[Evaluator] final fail - no retries left; showing last draft")
                print("[Evaluator] gaps:", ev.gaps)
                display(Markdown(report.markdown_report))
                print("[Pipeline] stopped (evaluator never passed)")
                return

            print("[Pipeline] retrying with new searches...")
            last_eval = ev



### Demo: clear question


In [ ]:
# Clear question - full pipeline
await deep_research(
    "What is Banoffe Pie? How to prepare one? How many calories does it bring?"
)


### Demo: ambiguous question
**Human-in-the-loop:** If triage is ambiguous, the notebook prints one follow-up question and **returns**. Rerun the same function with the same query and a **longer** user_clarifications string. Append blocks in this form (you can stack several over successive runs).

In [ ]:
# Vague question - expect printed clarification + return; then rerun with second arg, e.g.:
# await deep_research(
#     "Tell me about the best one",
#     """### Clarification
# **Q:** What specific category or context are you referring to when you say 'the best thing'?
# **A:** Best http python framework in 2026."""
# )
await deep_research(
    "Tell me about the best thing"
)
